# Mortgage Applications in the United States

This notebook addresses a binary classification challenge based on public HMDA mortgage application data released by the CFPB/FFIEC. Each record describes a U.S. home-purchase loan application, combining loan terms, property and neighborhood characteristics, financial information, and self-reported demographic attributes.

The goal is to estimate whether an application is accepted **(accepted = 1)** or denied **(accepted = 0)** from 56 explanatory variables. The training data contains roughly 105,000 labeled rows, while the test set contains about 45,000 rows.

Because most applications are accepted, the target is imbalanced. Performance is measured with **AUC ROC**, which evaluates the relative ranking of predicted probabilities and does not require selecting a decision threshold.

## 1. Imports

In [1]:
# Imports
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from class_utils import*

# Global Config
warnings.filterwarnings("ignore")
TRAIN_PATH, TEST_PATH = "data_clas/train.csv", "data_clas/test.csv"
MODEL_SIZE = "small"

print("Device:", DEVICE)

Device: cuda


## 2. Data

An 80/20 stratified train/validation split is used. All preprocessing steps: feature engineering, imputation, scaling, and encoding—are fitted exclusively on the training rows. Two feature representations are then constructed:

* **scl:** imputed, scaled, and one-hot encoded features, used for linear models, tree-based methods, boosting, and MLPs.
* **ord:** cleaned numeric features that retain missing values, together with ordinally encoded categoricals, tailored to foundation models.

### 2.1 Train

In [2]:
# Load train data
df = pd.read_csv(TRAIN_PATH)
y = df.pop(TARGET).to_numpy(np.int64)
df.pop("id")

# Split and preprocess
i_tr, i_val = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=y)
prep = Preprocessor(SELECTED_VARS).fit(df.iloc[i_tr])
X = prep.transform_views(df)
X_tr, X_val = take(X, i_tr), take(X, i_val)
y_tr, y_val = y[i_tr], y[i_val]

# Summary
print(f"Train: {len(i_tr):,} rows | Validation: {len(i_val):,} rows | "
      f"Features: {X['scl'].shape[1]} (scl) / {X['ord'].shape[1]} (ord) | Accepted: {y.mean():.1%}")

Train: 84,000 rows | Validation: 21,000 rows | Features: 158 (scl) / 34 (ord) | Accepted: 85.0%


### 2.2 Test

In [3]:
# Load and preprocess test data
test = pd.read_csv(TEST_PATH)
test_ids = test.pop("id")
test = test.drop(columns=[TARGET], errors="ignore")
X_test = prep.transform_views(test)

## 3. Model Training
Each model is fitted on the training set, its validation probabilities are stored in **P_val**, and the fitted estimator is kept in **models**.

In [4]:
models, P_val = {}, {}

### 3.1 Baseline
Predicts the training class prior for every row and serves as the reference for all other models.

In [5]:
# Define Model
models["Baseline"] = DummyClassifier(strategy="prior")

# Fit and Predict
P_val["Baseline"] = fit_predict(models["Baseline"], X_tr, y_tr, X_val)

### 3.2 Logistic Regression
A linear classifier that estimates class probabilities by applying the logistic function to a linear combination of features.

In [6]:
# Define Model
models["LogReg"] = LogisticRegression(max_iter=2000)

# Fit and Predict
P_val["LogReg"] = fit_predict(models["LogReg"], X_tr, y_tr, X_val)

### 3.3 Random Forest
A bagging ensemble of 1,000 decision trees, each trained on a bootstrap sample with random feature subsets per split.

In [7]:
# Define Model
models["RandomForest"] = RandomForestClassifier(n_estimators=1000, min_samples_leaf=5, n_jobs=-1, random_state=SEED)

# Fit and Predict
P_val["RandomForest"] = fit_predict(models["RandomForest"], X_tr, y_tr, X_val)

### 3.4 Extra Trees
An ensemble of 300 extremely randomized trees: unlike Random Forest, split thresholds are drawn at random and no bootstrap sampling is used.

In [8]:
# Define Model
models["ExtraTrees"] = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED)

# Fit and Predict
P_val["ExtraTrees"] = fit_predict(models["ExtraTrees"], X_tr, y_tr, X_val)

### 3.5 Histogram Gradient Boosting
Gradient-boosted trees trained on histogram-binned features. Predictions average three seeds, each using early stopping.

In [9]:
# Define hyperparameters
HGB_PARAMS = dict(learning_rate=0.02, max_leaf_nodes=127, min_samples_leaf=10, max_features=0.5, l2_regularization=5.0,
                  max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=30)

# Define Model
models["HistGB"] = SeedBag(HGB_PARAMS, n_seeds=3)

# Fit and Predict
P_val["HistGB"] = fit_predict(models["HistGB"], X_tr, y_tr, X_val)

### 3.6 XGBoost
Gradient-boosted trees using a second-order loss approximation, histogram-based splits, and leaf-wise growth.

In [10]:
# Define Model
models["XGBoost"] = EarlyStopped(
    XGBClassifier(n_estimators=2000, learning_rate=0.03, tree_method="hist", grow_policy="lossguide", max_depth=0,
                  max_leaves=127, min_child_weight=2, colsample_bynode=0.5, reg_lambda=5.0, eval_metric="auc",
                  early_stopping_rounds=50, n_jobs=-1, random_state=SEED),
    fit_kw=lambda: dict(verbose=False))

# Fit and Predict
P_val["XGBoost"] = fit_predict(models["XGBoost"], X_tr, y_tr, X_val)

### 3.7 LightGBM
A gradient boosting framework that builds trees leaf-wise and uses histogram-binned features, making it fast and memory-efficient on large tabular datasets.

In [11]:
# Define Model
models["LightGBM"] = EarlyStopped(
    LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=127, min_child_samples=10,
                   feature_fraction_bynode=0.5, reg_lambda=5.0, n_jobs=-1, random_state=SEED, verbose=-1),
    fit_kw=lambda: dict(eval_metric="auc", callbacks=[lgb.early_stopping(50, verbose=False)]))

# Fit and Predict
P_val["LightGBM"] = fit_predict(models["LightGBM"], X_tr, y_tr, X_val)

### 3.8 Multilayer Perceptron 
A feedforward neural network with two hidden layers, each combining a linear transformation, batch normalization, sigmoid activation, and dropout. It is trained with Adam and binary cross-entropy.

In [12]:
# Define Model
models["MLP"] = TorchMLP(hidden=(128, 64), dropout=0.1, lr=1e-3, batch=512, max_epochs=100, patience=10)

# Fit and Predict
P_val["MLP"] = fit_predict(models["MLP"], X_tr, y_tr, X_val)
print("Best epoch:", models["MLP"].best_epoch_)

  0%|          | 0/100 [00:00<?, ?epoch/s]

Best epoch: 26


## 4. Foundation Models
Foundation models are pretrained tabular transformers used here as in-context learners: no gradient updates are performed on the dataset. Instead, labeled training rows serve as context, and predictions are obtained through a forward pass. These models consume the **ord** feature view, with the context capped at **max_context** rows and queries scored in batches.

In [13]:
# Small batches
s_batch = dict(cat_idx=prep.ord_cat_idx, batch_rows=10000)

# Big batches
b_batch = dict(cat_idx=prep.ord_cat_idx, batch_rows=100000)

### 4.1 TabPFN-3.5
A Prior Labs transformer pretrained on synthetic tabular datasets, designed to approximate the Bayesian posterior predictive in a single forward pass.

In [14]:
# Define Model and Predict
run_model("TabPFN-3.5", TabPFNModel(version="3.5", size=MODEL_SIZE, max_context=10000, **s_batch),
          models, P_val, X_tr, y_tr, X_val)

### 4.2 TabICLv2
A transformer for tabular in-context learning, pretrained on synthetic datasets. It embeds features column-wise, models interactions across rows, and then performs in-context prediction.

In [15]:
# Define Model and Predict
run_model("TabICLv2", TabICLModel(n_estimators=4, max_context=25000, **s_batch),
          models, P_val, X_tr, y_tr, X_val)

### 4.3 Kumo-Tabular

A NVIDIA foundation model for tabular data. It makes predictions directly from the training rows it is given, without any gradient updates.

In [16]:
# Define Model and Predict
run_model("Kumo-Tabular", KumoTabularModel(size=MODEL_SIZE, num_estimators=16, max_context=None, **b_batch),
          models, P_val, X_tr, y_tr, X_val)

## 5. Results
Validation metrics are reported below. Threshold-dependent metrics are computed at each model’s MCC-optimal threshold, whereas PR-AUC is evaluated for the rejected class.

In [17]:
# Results Table
results, THR = results_table(P_val, y_val)
results

,AUC,PR-AUC (rejected),MCC,F1 macro,Bal. acc.,Accuracy,Threshold
Kumo-Tabular,0.9237,0.8322,0.7454,0.8698,0.8414,0.9388,0.57
HistGB,0.9195,0.8202,0.7279,0.8592,0.8243,0.9354,0.51
XGBoost,0.9187,0.8191,0.7272,0.8587,0.8235,0.9352,0.51
LightGBM,0.9186,0.8185,0.7282,0.8612,0.8335,0.9348,0.58
RandomForest,0.9143,0.8063,0.7107,0.8530,0.8283,0.9304,0.60
TabPFN-3.5,0.9140,0.8094,0.7134,0.8531,0.8226,0.9317,0.56
TabICLv2,0.9138,0.8105,0.7147,0.8533,0.8216,0.9321,0.54
ExtraTrees,0.9084,0.7979,0.7005,0.8490,0.8310,0.9270,0.66
MLP,0.9053,0.7955,0.7002,0.8447,0.8092,0.9293,0.51
LogReg,0.8914,0.7549,0.6513,0.8210,0.7895,0.9180,0.59


## 6. Test Predictions

The model with the highest validation AUC is refitted on the combined training and validation data, then applied to the test set using its validation threshold.

In [18]:
# Best model
best = results.drop("Baseline")["AUC"].idxmax()
print(f"Selected model: {best} | validation AUC: {results.loc[best, 'AUC']:.4f} | threshold: {THR[best]:.2f}")

# Refit and Predict
p_test = refit_predict(models, best, stack_views(X_tr, X_val), np.concatenate([y_tr, y_val]), X_test)
pred = pd.DataFrame({"id": test_ids, TARGET: p_test.round(4)})

# Save submission
pred.to_csv("class_pred.csv", index=False)
print(f"Saved class_pred.csv: {len(pred):,} rows | predicted acceptance rate: {pred[TARGET].mean():.1%}")

# Preview
pred.head()

Selected model: Kumo-Tabular | validation AUC: 0.9237 | threshold: 0.57
Saved class_pred.csv: 45,000 rows | predicted acceptance rate: 84.9%


,id,accepted
0,923582,0.9686
1,801040,1.0000
2,705186,0.1061
3,614701,0.9344
4,951449,0.9649
